# Movie Dataset Project - Part 1
**Student:** Neta Or Shaul  
**ID:** 323130716

**GitHub:** [View Project on GitHub](https://github.com/NetaShaul/file-ptoject)

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib.request
import os
import re
import time
import json
import numpy as np
import html


Downloading IMDb datasets while skipping already existing files to save time.

In [2]:
# Defining the URLs for the four required files according to requirements
files_to_download = {
    "title_basics": "https://datasets.imdbws.com/title.basics.tsv.gz",
    "title_ratings": "https://datasets.imdbws.com/title.ratings.tsv.gz",
    "title_principals": "https://datasets.imdbws.com/title.principals.tsv.gz",
    "name_basics": "https://datasets.imdbws.com/name.basics.tsv.gz"
}

# Creating a data directory if it doesn't already exist
if not os.path.exists('imdb_data'):
    os.makedirs('imdb_data')

# Download loop using urllib as practiced in the course
for file_id, url in files_to_download.items():
    destination = f"imdb_data/{file_id}.tsv.gz"
    if not os.path.exists(destination):
        print(f"Downloading {file_id}...")
        # This function performs a HTTP GET and saves the stream directly to a file
        urllib.request.urlretrieve(url, destination)
        print(f"Finished: {file_id}")
    else:
        print(f"File {file_id} already exists, skipping download.")

File title_basics already exists, skipping download.
File title_ratings already exists, skipping download.
File title_principals already exists, skipping download.
File name_basics already exists, skipping download.


Data Loading and Filtering
Loading the IMDb dataset and filtering for:
* **Type:** Feature films ('movie') between 60-300 minutes.
* **Year:** Released up to 2024.
* **Scope:** Titles starting with 'U' or 'V' (as assigned).

In [3]:
# Loading the dataset
df_basics = pd.read_csv("imdb_data/title_basics.tsv.gz", sep='\t', na_values='\\N', low_memory=False)

# Converting to numeric types (invalid values will automatically become NaN)
df_basics['startYear'] = pd.to_numeric(df_basics['startYear'], errors='coerce')
df_basics['runtimeMinutes'] = pd.to_numeric(df_basics['runtimeMinutes'], errors='coerce')

# Defining filtering conditions (Type, Year, Duration, and Titles starting with U or V)
condition = (
    (df_basics['titleType'] == 'movie') & 
    (df_basics['startYear'] <= 2024) & 
    (df_basics['runtimeMinutes'] >= 60) & 
    (df_basics['runtimeMinutes'] <= 300) &
    (df_basics['primaryTitle'].str.upper().str.startswith(('U', 'V'), na=False))
)

# Creating the filtered DataFrame without dropping NULLs
df_filtered = df_basics[condition].copy()

final_count = len(df_filtered)

# Defining the final variable for further processing
final_df = df_filtered

In [4]:
df_filtered

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
6624,tt0006709,movie,Under Suspicion,The Game of Liberty,0,1916.0,NaN,60.0,"Comedy,Crime"
6816,tt0006903,movie,Victory of Love,Kärleken segrar,0,1916.0,NaN,62.0,Drama
7397,tt0007498,movie,Under Two Flags,Under Two Flags,0,1916.0,NaN,60.0,"Adventure,Drama"
7421,tt0007522,movie,Vingarne,Vingarne,0,1916.0,NaN,69.0,Drama
9604,tt0009742,movie,Under Four Flags,Under Four Flags,0,1918.0,NaN,70.0,"Documentary,War"
...,...,...,...,...,...,...,...,...,...
12478411,tt9900908,movie,Useless Handcuffs,Tejô muyô,0,1969.0,NaN,89.0,"Action,Comedy,Crime"
12480870,tt9906218,movie,Unstoppable,Unstoppable,0,2019.0,NaN,84.0,Documentary
12482087,tt9908764,movie,Ue kara shita kara,Ue kara shita kara,0,1981.0,NaN,70.0,NaN
12482491,tt9909716,movie,Ushiro muki,Ushiro muki,0,1981.0,NaN,69.0,NaN


In [5]:
columns_to_keep = ['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']
df_filtered = df_filtered[columns_to_keep]
df_filtered

,tconst,primaryTitle,startYear,runtimeMinutes,genres
6624,tt0006709,Under Suspicion,1916.0,60.0,"Comedy,Crime"
6816,tt0006903,Victory of Love,1916.0,62.0,Drama
7397,tt0007498,Under Two Flags,1916.0,60.0,"Adventure,Drama"
7421,tt0007522,Vingarne,1916.0,69.0,Drama
9604,tt0009742,Under Four Flags,1918.0,70.0,"Documentary,War"
...,...,...,...,...,...
12478411,tt9900908,Useless Handcuffs,1969.0,89.0,"Action,Comedy,Crime"
12480870,tt9906218,Unstoppable,2019.0,84.0,Documentary
12482087,tt9908764,Ue kara shita kara,1981.0,70.0,NaN
12482491,tt9909716,Ushiro muki,1981.0,69.0,NaN


In [6]:
# Loading the ratings dataset
df_ratings = pd.read_csv("imdb_data/title_ratings.tsv.gz", sep='\t', na_values='\\N')

# Merging ratings with our filtered movie list
# We use a 'left' join to ensure we keep all movies from our filtered set
final_df = pd.merge(df_filtered, df_ratings, on='tconst', how='left')

# Previewing the merged data
final_df.head()

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0006709,Under Suspicion,1916.0,60.0,"Comedy,Crime",7.2,34.0
1,tt0006903,Victory of Love,1916.0,62.0,Drama,NaN,NaN
2,tt0007498,Under Two Flags,1916.0,60.0,"Adventure,Drama",6.5,30.0
3,tt0007522,Vingarne,1916.0,69.0,Drama,5.7,230.0
4,tt0009742,Under Four Flags,1918.0,70.0,"Documentary,War",NaN,NaN


**Processing Lead Actors**

Processing the `title.principals` dataset using a chunk-based approach to identify the primary cast. This stage involves filtering actors for the selected movies and retaining the top 5 leads based on their ranking.

In [7]:
my_movie_ids = set(final_df['tconst'])
len(my_movie_ids)

11121

**Chunk-based Data Processing**

Reading the large `title.principals` file in chunks to efficiently filter for relevant actors while maintaining low memory usage.

In [8]:
# --- Defining Chunk-based Reading and Initial Filtering ---

path_principals = "imdb_data/title_principals.tsv.gz"
chunk_size = 1000000  
filtered_list = []    

print("Starting to process the principals file in chunks...")

# Processing the file in chunks to prevent memory crashes
for chunk in pd.read_csv(path_principals, sep='\t', na_values='\\N', chunksize=chunk_size, low_memory=False, 
                         usecols=['tconst', 'nconst', 'category', 'ordering']):
    

    # Filtering for actors/actresses from the selected movie IDs
    my_movie_mask = (chunk['tconst'].isin(my_movie_ids)) & (chunk['category'].isin(['actor', 'actress']))
    
    # Append the filtered rows to our list
    filtered_list.append(chunk[my_movie_mask])

print("Successfully processed all chunks and kept relevant actors.")


Starting to process the principals file in chunks...
Successfully processed all chunks and kept relevant actors.


Combining the filtered results and ranking actors by their importance to select the top 5 leads for each movie.

In [9]:
# Consolidate chunks and sort by movie and actor ranking
df_principals_small = pd.concat(filtered_list, ignore_index=True)
df_principals_small.sort_values(['tconst', 'ordering'], inplace=True)

# Select top 5 leads and aggregate their IDs into a single string
top_5_ids = df_principals_small.groupby('tconst').head(5)
actors_grouped = top_5_ids.groupby('tconst')['nconst'].apply(lambda x: ', '.join(x)).reset_index()

actors_grouped.columns = ['tconst', 'lead_actors_ids']
print("Ranking and aggregation completed.")
top_5_ids.head()

Ranking and aggregation completed.


,tconst,ordering,nconst,category
0,tt0006709,1,nm0024706,actor
1,tt0006709,2,nm0613115,actor
2,tt0006709,3,nm0184766,actress
3,tt0006709,4,nm0944017,actor
4,tt0006709,5,nm0265498,actress


In [10]:
# --- Final Merge with the Central DataFrame ---

# Remove column if it exists to prevent duplicates
if 'lead_actors_ids' in final_df.columns:
    final_df.drop(columns=['lead_actors_ids'], inplace=True)

# Integrate actor information using a left join
final_df = pd.merge(final_df, actors_grouped, on='tconst', how='left')

print("-" * 30)
print(f"Merge complete!")
print(f"Final dataset size: {len(final_df)} movies.")
print("-" * 30)

# Display the updated DataFrame
final_df.head()

------------------------------
Merge complete!
Final dataset size: 11121 movies.
------------------------------


,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0006709,Under Suspicion,1916.0,60.0,"Comedy,Crime",7.2,34.0,"nm0024706, nm0613115, nm0184766, nm0944017, nm..."
1,tt0006903,Victory of Love,1916.0,62.0,Drama,NaN,NaN,"nm0436013, nm0459320, nm0054011, nm0188850, nm..."
2,tt0007498,Under Two Flags,1916.0,60.0,"Adventure,Drama",6.5,30.0,"nm0000847, nm0382229, nm0392059, nm0923657, nm..."
3,tt0007522,Vingarne,1916.0,69.0,Drama,5.7,230.0,"nm0251622, nm0064949, nm0361319, nm0492280, nm..."
4,tt0009742,Under Four Flags,1918.0,70.0,"Documentary,War",NaN,NaN,NaN


In [11]:
# List of columns to be added for external data integration
cols_to_add = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']

# Initialize columns with None if they do not already exist in the dataframe
for col in cols_to_add:
    if col not in final_df.columns:
        final_df[col] = None

print(f"Added {len(cols_to_add)} new columns to the dataframe.")

Added 5 new columns to the dataframe.


In [12]:
final_df

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,budget,BoxOffice,plot
0,tt0006709,Under Suspicion,1916.0,60.0,"Comedy,Crime",7.2,34.0,"nm0024706, nm0613115, nm0184766, nm0944017, nm...",None,None,None,None,None
1,tt0006903,Victory of Love,1916.0,62.0,Drama,NaN,NaN,"nm0436013, nm0459320, nm0054011, nm0188850, nm...",None,None,None,None,None
2,tt0007498,Under Two Flags,1916.0,60.0,"Adventure,Drama",6.5,30.0,"nm0000847, nm0382229, nm0392059, nm0923657, nm...",None,None,None,None,None
3,tt0007522,Vingarne,1916.0,69.0,Drama,5.7,230.0,"nm0251622, nm0064949, nm0361319, nm0492280, nm...",None,None,None,None,None
4,tt0009742,Under Four Flags,1918.0,70.0,"Documentary,War",NaN,NaN,NaN,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11116,tt9900908,Useless Handcuffs,1969.0,89.0,"Action,Comedy,Crime",NaN,NaN,"nm0441526, nm0297835, nm0297788, nm0766277, nm...",None,None,None,None,None
11117,tt9906218,Unstoppable,2019.0,84.0,Documentary,7.7,26.0,NaN,None,None,None,None,None
11118,tt9908764,Ue kara shita kara,1981.0,70.0,NaN,NaN,NaN,"nm10493162, nm3459781, nm10534879",None,None,None,None,None
11119,tt9909716,Ushiro muki,1981.0,69.0,NaN,NaN,NaN,"nm10535432, nm10535433, nm1000331",None,None,None,None,None


**Data Collection**

Retrieving missing details such as budget, box office, languages, and countries directly from Wikipedia to enrich the dataset.

In [13]:
def get_movie_data_combined_FINAL(title, year):
    """
    Scrapes Wikipedia for movie details using URL fallbacks and API search.
    """
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    try:
        clean_year = str(int(float(year)))
    except:
        clean_year = str(year)
    
    # Preparing URL variations
    original_format = title.replace(' ', '_')
    clean_title = title.title().replace(' ', '_')
    
    res = {'Language': 'N/A', 'Country': 'N/A', 'budget': 'N/A', 'BoxOffice': 'N/A', 'plot': 'N/A'}
    
    def scrape_from_url(target_url):
        try:
            req = urllib.request.Request(target_url, headers=headers)
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    page_soup = BeautifulSoup(resp.read(), 'html.parser')
                    found_data = {'Language': 'N/A', 'Country': 'N/A', 'budget': 'N/A', 'BoxOffice': 'N/A', 'plot': 'N/A'}
                    
                    # Extracting data from the Infobox table
                    infobox = page_soup.find('table', {'class': 'infobox'})
                    if infobox:
                        for row in infobox.find_all('tr'):
                            h, d = row.find('th'), row.find('td')
                            if h and d:
                                label = " ".join(h.text.split()).lower()
                                # Cleaning raw text from footnotes [1] and non-breaking spaces
                                value = d.get_text(separator=" ", strip=True).replace('\xa0', ' ')
                                value = re.sub(r'\[.*?\]', '', value).strip()
                                
                                if 'language' in label: found_data['Language'] = value
                                elif 'country' in label: found_data['Country'] = value
                                elif 'budg' in label: found_data['budget'] = value
                                elif 'box' in label or 'office' in label: found_data['BoxOffice'] = value
                    
                    # PLOT EXTRACTION: Scanning headers for plot-related keywords
                    keywords = ['plot', 'synopsis', 'summary', 'scenario', 'story', 'premise']
                    target_header = None
                    for h in page_soup.find_all(['h2', 'h3']):
                        if any(k in h.get_text().lower() for k in keywords):
                            target_header = h
                            break
                    
                    if target_header:
                        for element in target_header.find_all_next():
                            if element.name in ['h2', 'h3']: break
                            if element.name == 'p':
                                p_text = element.get_text(strip=True)
                                if len(p_text) > 30:
                                    found_data['plot'] = p_text
                                    break
                    return found_data
        except:
            return None

    # Step 1: Attempt data extraction using direct URL variations
    urls = [
        f"https://en.wikipedia.org/wiki/{original_format}_({clean_year}_film)",
        f"https://en.wikipedia.org/wiki/{clean_title}_({clean_year}_film)", 
        f"https://en.wikipedia.org/wiki/{original_format}",
        f"https://en.wikipedia.org/wiki/{clean_title}"
    ]
    
    data = None
    for url in urls:
        data = scrape_from_url(url)
        if data and (data['Language'] != 'N/A' or data['plot'] != 'N/A'): 
            break
    
    # Step 2: API Fallback - Search Wikipedia if direct links fail
    if not data or (data['Language'] == 'N/A' and data['plot'] == 'N/A'):
        try:
            search_q = f"{title} {clean_year} film".replace(' ', '%20')
            api_url = f"https://en.wikipedia.org/w/api.php?action=opensearch&search={search_q}&limit=1&format=json"
            with urllib.request.urlopen(urllib.request.Request(api_url, headers=headers)) as resp:
                search_data = json.loads(resp.read().decode())
                if search_data[3]: 
                    data = scrape_from_url(search_data[3][0])
        except:
            pass

    return data if data else res

**Data Extraction Loop with Backup**

Executing the scraping function across the dataset. This block includes a resume feature to continue from the last saved state and performs periodic backups to ensure progress is preserved during the process.

In [14]:

# 1. SETUP: Define file names for saving and resuming progress
backup_file = 'movies_data_backup.csv'
final_output = 'movies_data_final.csv'

# 2. CONTINUITY: Check if a backup exists to resume from the last saved state
if os.path.exists(backup_file):
    print(f"Loading existing backup: '{backup_file}'...")
    movies_scraped_df = pd.read_csv(backup_file)
else:
    print("Starting a new run from final_df...")
    movies_scraped_df = final_df.copy()

print(f"Processing {len(movies_scraped_df)} movies total.")

# 3. EXECUTION LOOP
for i, row in movies_scraped_df.iterrows():
    
    # Skip rows that are already completed
    if pd.notna(movies_scraped_df.at[i, 'plot']) and movies_scraped_df.at[i, 'plot'] not in ['N/A', 'None', None]:
        continue
        
    try:
        # DATA RETRIEVAL
        movie_info = get_movie_data_combined_FINAL(row['primaryTitle'], row['startYear'])
        
        # UPDATE
        movies_scraped_df.at[i, 'Language'] = movie_info['Language']
        movies_scraped_df.at[i, 'Country'] = movie_info['Country']
        movies_scraped_df.at[i, 'budget'] = movie_info['budget']
        movies_scraped_df.at[i, 'BoxOffice'] = movie_info['BoxOffice']
        movies_scraped_df.at[i, 'plot'] = movie_info['plot']
        
        # MONITORING: Clean progress log every 20 movies
        if i % 20 == 0:
            print(f"Index {i} | Movie: {row['primaryTitle']} | Status: Done")

        # BACKUP: Save progress quietly every 100 movies
        if i % 100 == 0 and i > 0:
            movies_scraped_df.to_csv(backup_file, index=False)
            print(f"--- Progress saved at index {i} ---")

        time.sleep(0.5) 

    except Exception as e:
        print(f"!!! Error at index {i} ({row['primaryTitle']}): {e}")
        time.sleep(5)
        continue

# 4. FINALIZATION
movies_scraped_df.to_csv(final_output, index=False)
print(f"\nAll done! Final dataset saved to: {final_output}")

Starting a new run from final_df...
Processing 11121 movies total.
Index 0 | Movie: Under Suspicion | Status: Done
Index 20 | Movie: Värmlänningarna | Status: Done
Index 40 | Movie: Up in Mabel's Room | Status: Done
Index 60 | Movie: Vad kvinnan vill | Status: Done
Index 80 | Movie: Up the River | Status: Done
Index 100 | Movie: Viennese Waltz | Status: Done
--- Progress saved at index 100 ---
Index 120 | Movie: Vad veta väl männen? | Status: Done
Index 140 | Movie: Under Pressure | Status: Done
Index 160 | Movie: Vratar | Status: Done
Index 180 | Movie: Under the Big Top | Status: Done
Index 200 | Movie: Union Pacific | Status: Done
--- Progress saved at index 200 ---
Index 220 | Movie: Vengeance of the Deep | Status: Done
Index 240 | Movie: Unfinished Business | Status: Done
Index 260 | Movie: Varaventtiili | Status: Done
Index 280 | Movie: Vi mötte stormen | Status: Done
Index 300 | Movie: Voodoo Man | Status: Done
--- Progress saved at index 300 ---
Index 320 | Movie: Victory of Wo

**Data Cleaning**

Resetting rows that have a plot but are missing all other details (Language, Country, Budget and Box Office). This removes inconsistent data where the scraping might have fetched the wrong page.

In [39]:
# Load the dataset
movies_table = pd.read_csv('movies_data_final.csv')

In [40]:


# Define the mask for suspect rows (Plot exists, but ALL movie-specific data is missing)
suspect_mask = (movies_table['plot'].notna()) & \
       (movies_table['Language'].isna()) & \
       (movies_table['Country'].isna()) & \
       (movies_table['budget'].isna()) & \
       (movies_table['BoxOffice'].isna())

# Print samples of what we are about to reset
print(f"Number of suspect rows to be reset: {suspect_mask.sum()}")
if suspect_mask.sum() > 0:
    print("\nSamples of rows that will be cleaned (Titles and Plot previews):")
    print(movies_table.loc[suspect_mask, ['primaryTitle', 'startYear', 'plot']].head(10))

# Execute the cleanup on the dataframe in memory
movies_table.loc[suspect_mask, ['plot', 'Language', 'Country', 'budget', 'BoxOffice']] = np.nan

print("\nCleanup in memory is done.")

Number of suspect rows to be reset: 220

Samples of rows that will be cleaned (Titles and Plot previews):
                 primaryTitle  startYear  \
43                    Volcano     1926.0   
115       Unfinished Symphony     1934.0   
126                  Voltaire     1933.0   
291           U-Boat Prisoner     1944.0   
412            Valley of Fire     1951.0   
483       Valley of the Kings     1954.0   
587           Valentine's Day     1959.0   
618              Voskreseniye     1960.0   
628  Ulysses Against Hercules     1962.0   
689              Via Margutta     1960.0   

                                                  plot  
43   Volcanoes are not distributed evenly over the ...  
115  In 1823, the Graz Music Society gave Schubert ...  
126  Voltaire had an enormous influence on the deve...  
291  TheGestaposends Gunther Rudehoff aboard a U-bo...  
412  Prehistoricinhabitants of the Valley of Fire i...  
483  The Theban Hills are dominated by the peak ofa...  
587  Numer

In [41]:
movies_table_1 = movies_table.copy()

**Targeted Data Recovery**

After identifying gaps in the dataset, a second scraping phase was implemented. Instead of re-processing the entire dataframe, a filter was applied to target only the rows with missing or inconsistent values. A more robust function was used to handle complex Wikipedia structures, ensuring a more complete and reliable final dataset.

In [151]:


def get_movie_data_final_clean(title, year):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    }
    
    formatted_title = title.replace(' ', '_')
    try:
        clean_year = str(int(float(year)))
    except:
        clean_year = str(year)
    
    urls = [
        f"https://en.wikipedia.org/wiki/{formatted_title}_({clean_year}_film)",
        f"https://en.wikipedia.org/wiki/{formatted_title}_(film)",
        f"https://en.wikipedia.org/wiki/{formatted_title}"
    ]
    
    for url in urls:
        try:
            resp = requests.get(url, headers=headers, timeout=5)
            if resp.status_code == 200:
                page_html_lower = resp.text.lower()
                
                # Safety indicators to confirm this is a movie page
                movie_indicators = [
                    'directed by', 'starring', 'produced by', 
                    'screenplay by', 'distributed by', 'cinematography',
                    'music by', 'filmography', 'release date'
                ]
                
                has_infobox = 'infobox' in page_html_lower
                is_confirmed_movie = any(indicator in page_html_lower for indicator in movie_indicators)
                
                if has_infobox and is_confirmed_movie:
                    page_html = resp.text
                    # Initialize results with np.nan for consistency
                    results = {
                        'url': url, 
                        'Language': np.nan, 
                        'Country': np.nan, 
                        'budget': np.nan, 
                        'BoxOffice': np.nan, 
                        'plot': np.nan
                    }
                    
                    # Updated patterns to handle plural "Countries" and "Languages"
                    patterns = {
                        'Language': r'Languages?</th><td.*?>(.*?)</td>',
                        'Country': r'(?:Country|Countries|Origin)</th><td.*?>(.*?)</td>',
                        'budget': r'Budget</th><td.*?>(.*?)</td>',
                        'BoxOffice': r'(?:Box office)</th><td.*?>(.*?)</td>'
                    }
                    
                    for key, pattern in patterns.items():
                        match = re.search(pattern, page_html, re.IGNORECASE | re.DOTALL)
                        if match:
                            val = html.unescape(match.group(1))
                            val = re.sub(r'<br\s*/?>', ', ', val) 
                            val = re.sub(r'<.*?>', '', val)       
                            val = re.sub(r'\[\d+\]', '', val)     
                            val = " ".join(val.split())           
                            results[key] = val.strip()
                    
                    # Extracting the Plot summary
                    plot_pattern = r'<(?:h2|h3).*?>(?:Plot|Synopsis|Summary)</(?:h2|h3)>.*?<p>(.*?)</p>'
                    plot_match = re.search(plot_pattern, page_html, re.IGNORECASE | re.DOTALL)
                    if plot_match:
                        p_text = html.unescape(plot_match.group(1))
                        p_text = re.sub(r'<.*?>', '', p_text)
                        p_text = re.sub(r'\[\d+\]', '', p_text)
                        results['plot'] = " ".join(p_text.split()).strip()
                    
                    return results
        except:
            continue
            
    return "Not Found"


In [18]:

# 1. Unify missing values
movies_table = movies_table.replace('N/A', np.nan)

# 2. Define the targets
target_mask = (movies_table['Language'].isna()) | \
              (movies_table['Country'].isna()) | \
              (movies_table['plot'].isna())

to_process = movies_table[target_mask]
print(f"Total rows to process: {len(to_process)} out of {len(movies_table)}")

# 3. Master Update Loop
count = 0
updated_count = 0

for index, row in to_process.iterrows():
    # Call the scraping function
    new_data = get_movie_data_final_clean(row['primaryTitle'], row['startYear'])
    
    if isinstance(new_data, dict):
        # Update empty fields only
        for field in ['Language', 'Country', 'budget', 'BoxOffice', 'plot']:
            if pd.isna(movies_table.at[index, field]):
                movies_table.at[index, field] = new_data.get(field, np.nan)
        updated_count += 1
    
    count += 1
    
    # Anti-blocking delay
    time.sleep(0.5) 
    
    # Clean status update every 50 rows
    if count % 50 == 0:
        print(f"Status: Checked {count}/{len(to_process)} | Successfully updated: {updated_count}")
        
    # Emergency Backup every 100 rows
    if count % 100 == 0:
        movies_table.to_csv('movies_checkpoint.csv', index=False)
        print(f"--- Checkpoint saved ---")

# 4. Final Save
movies_table.to_csv('movies_data_final_v2.csv', index=False)
print(f"Finish! Processed {count} rows. Total updated: {updated_count}")

Total rows to process: 9321 out of 11121
Status: Checked 50/9321 | Successfully updated: 25
Status: Checked 100/9321 | Successfully updated: 45
--- Checkpoint saved ---
Status: Checked 150/9321 | Successfully updated: 57
Status: Checked 200/9321 | Successfully updated: 66
--- Checkpoint saved ---
Status: Checked 250/9321 | Successfully updated: 84
Status: Checked 300/9321 | Successfully updated: 106
--- Checkpoint saved ---
Status: Checked 350/9321 | Successfully updated: 116
Status: Checked 400/9321 | Successfully updated: 136
--- Checkpoint saved ---
Status: Checked 450/9321 | Successfully updated: 153
Status: Checked 500/9321 | Successfully updated: 166
--- Checkpoint saved ---
Status: Checked 550/9321 | Successfully updated: 180
Status: Checked 600/9321 | Successfully updated: 194
--- Checkpoint saved ---
Status: Checked 650/9321 | Successfully updated: 206
Status: Checked 700/9321 | Successfully updated: 214
--- Checkpoint saved ---
Status: Checked 750/9321 | Successfully updated:

In [152]:
movies_after_df = pd.read_csv('movies_data_final_v2.csv')

**Scraping Performance Analysis**

This final check compares the dataset before and after the targeted recovery phase. By calculating the coverage of Wikipedia-sourced columns (Language, Country, Budget, Box Office, and Plot), we can measure the exact improvement in data density and verify the success of the scraping process.

In [153]:


def run_final_comparison(df_before, df_after):
    # Only evaluate columns sourced from Wikipedia
    wiki_cols = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']
    total_rows = len(df_after)
    
    # Total possible Wikipedia data points (rows * 5 columns)
    total_wiki_cells = total_rows * len(wiki_cols)
    
    results = {}
    data_versions = {
        'Before Update (movies_table)': df_before,
        'After Update (movies_after_df)': df_after
    }
    
    for name, df in data_versions.items():
        # 1. Count NULLs in Wikipedia columns
        null_counts = df[wiki_cols].isnull().sum()
        
        # 2. Row-based success metrics
        at_least_one = df[wiki_cols].notnull().any(axis=1).sum()
        perfect_rows = df[wiki_cols].notnull().all(axis=1).sum()
        
        # 3. Calculate coverage percentage specifically for Wikipedia data
        total_wiki_nulls = null_counts.sum()
        wiki_coverage_pct = ((total_wiki_cells - total_wiki_nulls) / total_wiki_cells) * 100
        
        results[name] = {
            'Language_NULLs': null_counts['Language'],
            'Country_NULLs': null_counts['Country'],
            'Budget_NULLs': null_counts['budget'],
            'BoxOffice_NULLs': null_counts['BoxOffice'],
            'Plot_NULLs': null_counts['plot'],
            'At_Least_One_Wiki_Data': at_least_one,
            'Perfect_Wiki_Rows_5/5': perfect_rows,
            'Overall_Wiki_Coverage_%': round(wiki_coverage_pct, 2) 
        }
    
    # Create comparison table
    comparison_df = pd.DataFrame(results).T
    
    print(f"--- Scraping Performance: Wikipedia Data Only (Total Movies: {total_rows}) ---")
    display(comparison_df)

# Execute
run_final_comparison(movies_table_1, movies_after_df)

--- Scraping Performance: Wikipedia Data Only (Total Movies: 11121) ---


,Language_NULLs,Country_NULLs,Budget_NULLs,BoxOffice_NULLs,Plot_NULLs,At_Least_One_Wiki_Data,Perfect_Wiki_Rows_5/5,Overall_Wiki_Coverage_%
Before Update (movies_table),8320.0,8633.0,10582.0,10441.0,8989.0,2891.0,322.0,15.54
After Update (movies_after_df),8198.0,8286.0,10542.0,10372.0,8816.0,3062.0,395.0,16.89


**Optimized Data Recovery with OMDB API**

To make the dataset as complete as possible while staying within the OMDB API limit of 1,000 requests per day, I'm using a targeted hierarchical approach. I prioritized rows based on their data density starting with those missing only a plot, then those missing one or two additional fields. This ensures that the limited API calls are spent on the most recoverable movies, using the unique IMDb ID (tconst) to guarantee perfect data alignment.

In [53]:
# Create a mask for rows where 'plot' is specifically missing
plot_missing_mask = movies_after_df['plot'].isnull()
wiki_cols = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']

# Calculate how many other Wiki columns are missing for those rows
# We use wiki_cols but exclude 'plot' from the sum calculation
other_missing_count = movies_after_df.loc[plot_missing_mask, wiki_cols].isnull().drop(columns='plot').sum(axis=1)

# Categorize and print results
print(f"1. Missing ONLY Plot: {(other_missing_count == 0).sum()}")
print(f"2. Missing Plot + 1 other: {(other_missing_count == 1).sum()}")
print(f"3. Missing Plot + 2 others: {(other_missing_count == 2).sum()}")


1. Missing ONLY Plot: 14
2. Missing Plot + 1 other: 56
3. Missing Plot + 2 others: 568


In [56]:
OMDB_API_KEY = 'd13938b3' 

def get_omdb_plot(tconst):
    """Fetches the plot for a specific IMDb ID from OMDB API."""
    url = f"http://www.omdbapi.com/?i={tconst}&plot=full&apikey={OMDB_API_KEY}"
    try:
        response = requests.get(url).json()
        if response.get('Response') == 'True':
            plot = response.get('Plot')
            # Only return if it's actual text and not "N/A"
            return plot if plot and plot != 'N/A' else None
    except:
        return None
    return None

In [55]:
# 1. Target rows missing a plot but having at least 2 other metrics
other_cols = [c for c in wiki_cols if c != 'plot']


filled_others_count = movies_after_df[other_cols].notnull().sum(axis=1)

target_mask = (movies_after_df['plot'].isnull()) & (filled_others_count >= 2)


indices_to_fix = movies_after_df[target_mask].index

print(f"Targeting {len(indices_to_fix)} valuable movies (missing plot but have other data).")

# 2. Update loop with OMDB API
updated_count = 0
for idx in indices_to_fix:
    # Optional: Safety break if you want to limit to 1000 calls per day
    if updated_count >= 1000:
        print("Reached daily API limit (1000). Stopping...")
        break
        
    tconst = movies_after_df.at[idx, 'tconst']
    new_plot = get_omdb_plot(tconst)
    
    if new_plot:
        movies_after_df.at[idx, 'plot'] = new_plot
        updated_count += 1
        
    if updated_count % 100 == 0 and updated_count > 0:
        print(f"Progress: {updated_count} plots filled...")

print(f"Finished! Added {updated_count} plots to your dataset.")

Targeting 638 valuable movies (missing plot but have other data).
Progress: 100 plots filled...
Progress: 200 plots filled...
Progress: 200 plots filled...
Progress: 300 plots filled...
Progress: 400 plots filled...
Progress: 500 plots filled...
Finished! Added 514 plots to your dataset.


In [57]:
# Saving the updated data to a physical CSV file
movies_after_df.to_csv('movies_final_after_omdb.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [132]:
v1_initial_wiki = movies_table_1.copy()           # Original data
v2_update_wiki = pd.read_csv('movies_data_final_v2.csv') # After 2nd Wiki scraping
v3_update_omdb = pd.read_csv('movies_final_after_omdb.csv')          # Final enrichment via OMDB API


**Multi-Stage Data Enrichment Progress**

This final report provides a comprehensive comparison between the three versions of our dataset: the initial Wikipedia scrape, the refined Wikipedia recovery, and the final OMDB integration. By tracking the reduction in missing values and the increase in perfect records, we can quantify the total improvement in data quality throughout the project.

In [133]:
def generate_final_perfect_report(v1, v2, v3):
    wiki_cols = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']
    # Internal function to calculate basic metrics
    def get_stats(df):
        nulls = df[wiki_cols].isnull().sum()
        perfect = (df[wiki_cols].notnull().all(axis=1)).sum()
        at_least_one = (df[wiki_cols].notnull().any(axis=1)).sum()
        return nulls, perfect, at_least_one

    s1_n, s1_p, s1_any = get_stats(v1)
    s2_n, s2_p, s2_any = get_stats(v2)
    s3_n, s3_p, s3_any = get_stats(v3)
    
    # Part 1: Tracking the reduction of missing values
    report = pd.DataFrame({
        'V1_Initial_Wiki': s1_n,
        'V2_Wiki_Update': s2_n,
        'V3_After_OMDB': s3_n
    })
    report.index = [f"{col} (Missing)" for col in report.index]
    
    # Calculating the deltas (improvement) for each stage
    report['Wiki_Reduction'] = report['V1_Initial_Wiki'] - report['V2_Wiki_Update']
    report['OMDB_Reduction'] = report['V2_Wiki_Update'] - report['V3_After_OMDB']
    report['Total_Filled'] = report['V1_Initial_Wiki'] - report['V3_After_OMDB']
    
    # Part 2: Quality benchmarks (Perfect rows vs. partial coverage)
    summary = pd.DataFrame({
        'V1_Initial_Wiki': [s1_p, s1_any],
        'V2_Wiki_Update': [s2_p, s2_any],
        'V3_After_OMDB': [s3_p, s3_any],
        'Wiki_Reduction': [s2_p - s1_p, s2_any - s1_any],
        'OMDB_Reduction': [s3_p - s2_p, s3_any - s2_any],
        'Total_Filled': [s3_p - s1_p, s3_any - s1_any]
    }, index=['Perfect Rows (5/5)', 'Coverage (1+ Detail)'])
    # Combine everything into one master report
    final = pd.concat([report, summary])
    
    print("---  DATA ENRICHMENT PROGRESS REPORT ---")
    print("For missing values, a positive Delta shows how many gaps were filled. For quality metrics, it shows the increase in complete records.")
    return final

# Execute the final performance report
display(generate_final_perfect_report(v1_initial_wiki, v2_update_wiki, v3_update_omdb))

---  DATA ENRICHMENT PROGRESS REPORT ---
For missing values, a positive Delta shows how many gaps were filled. For quality metrics, it shows the increase in complete records.


,V1_Initial_Wiki,V2_Wiki_Update,V3_After_OMDB,Wiki_Reduction,OMDB_Reduction,Total_Filled
Language (Missing),8320,8198,8198,122,0,122
Country (Missing),8633,8286,8286,347,0,347
budget (Missing),10582,10542,10542,40,0,40
BoxOffice (Missing),10441,10372,10372,69,0,69
plot (Missing),8989,8816,8302,173,514,687
Perfect Rows (5/5),322,395,409,73,14,87
Coverage (1+ Detail),2891,3062,3062,171,0,171


**Data Type Conversion & Cleaning**

After consolidating the data from all sources, this step focuses on converting the columns to their correct data types. We are transforming strings into numeric values (for Budget and Box Office) and ensuring dates are properly formatted, turning the raw text into a dataset ready for analysis.

In [134]:
print("--- Data Types for V3_Update_OMDB ---")
display(v3_update_omdb.dtypes)

--- Data Types for V3_Update_OMDB ---


tconst              object
primaryTitle        object
startYear          float64
runtimeMinutes     float64
genres              object
averageRating      float64
numVotes           float64
lead_actors_ids     object
Language            object
Country             object
budget              object
BoxOffice           object
plot                object
dtype: object

**Primary Language Extraction**

To meet the project requirements, the primary language is extracted for each film. This process standardizes multi-language entries by selecting the first listed language, unifying "Silent" films, and removing auxiliary information in parentheses. This ensuring a clean categorical variable for subsequent statistical analysis.

In [135]:
def extract_primary_language(lang_str):
    if pd.isna(lang_str) or lang_str == 'None':
        return np.nan
    
    primary = str(lang_str).split(',')[0].split('/')[0].strip()
    
    if 'silent' in primary.lower():
        return 'Silent'
    
    primary = re.sub(r'\(.*?\)', '', primary).strip()
    
    return primary

# Applying the extraction
v3_update_omdb['Language'] = v3_update_omdb['Language'].apply(extract_primary_language)

# Check the results
print("--- Primary Language Extraction Complete ---")
v3_update_omdb[['primaryTitle', 'Language']].head(10)

--- Primary Language Extraction Complete ---


,primaryTitle,Language
0,Under Suspicion,Silent
1,Victory of Love,NaN
2,Under Two Flags,Silent
3,Vingarne,Silent
4,Under Four Flags,NaN
5,Vengeance,English
6,Virtuous Wives,Silent
7,Vor tids helte,NaN
8,Unknown Love,NaN
9,Victory,Silent


**Data Type Verification**

A deep check is performed on key columns to confirm that all categorical and text based data is correctly stored as strings. This validation step ensures consistency across the dataset and prevents errors during subsequent text processing and analysis.

In [136]:
# List of columns to verify
cols_to_check = ['tconst', 'primaryTitle', 'Language', 'Country', 'plot']

print("--- Data Type Verification per Column ---")

for col in cols_to_check:
    # Accessing the first item in the column (index 0)
    first_val = v3_update_omdb[col].iloc[0]
    
    # Printing the result in a clear, single line per column
    print(f"Deep Check on '{col}' column - Type of the first value: {type(first_val)}")

--- Data Type Verification per Column ---
Deep Check on 'tconst' column - Type of the first value: <class 'str'>
Deep Check on 'primaryTitle' column - Type of the first value: <class 'str'>
Deep Check on 'Language' column - Type of the first value: <class 'str'>
Deep Check on 'Country' column - Type of the first value: <class 'str'>
Deep Check on 'plot' column - Type of the first value: <class 'str'>


**Data Transformation: String to List**

As required for the project, the 'lead_actors_ids' and 'genres' columns have been converted from strings into Python lists to meet the specified data structure requirements.

In [137]:
# Convert lead_actors_ids column from string to list, or NaN if empty
v3_update_omdb['lead_actors_ids'] = v3_update_omdb['lead_actors_ids'].apply(
    lambda x: x.split(',') if isinstance(x, str) and x.strip() != "" else np.nan
)

# Convert genres column from string to list, or NaN if empty
v3_update_omdb['genres'] = v3_update_omdb['genres'].apply(
    lambda x: x.split(',') if isinstance(x, str) and x.strip() != "" else np.nan
)

# Verification and logging the change
print(f"Data conversion complete: Columns changed from String to List (Empty values set to NaN).")
print(f"New type for lead_actors_ids: {type(v3_update_omdb['lead_actors_ids'].iloc[0])}")
print(f"New type for genres: {type(v3_update_omdb['genres'].iloc[0])}")

v3_update_omdb[['primaryTitle', 'lead_actors_ids', 'genres']].head()

Data conversion complete: Columns changed from String to List (Empty values set to NaN).
New type for lead_actors_ids: <class 'list'>
New type for genres: <class 'list'>


,primaryTitle,lead_actors_ids,genres
0,Under Suspicion,"[nm0024706, nm0613115, nm0184766, nm0944017...","[Comedy, Crime]"
1,Victory of Love,"[nm0436013, nm0459320, nm0054011, nm0188850...",[Drama]
2,Under Two Flags,"[nm0000847, nm0382229, nm0392059, nm0923657...","[Adventure, Drama]"
3,Vingarne,"[nm0251622, nm0064949, nm0361319, nm0492280...",[Drama]
4,Under Four Flags,NaN,"[Documentary, War]"


**Numeric Type Conversion**

To meet the project requirements, the 'startYear', 'runtimeMinutes', and 'numVotes' columns have been converted to Integers.

In [138]:
# Convert Year, Minutes and Votes to Integer (Int64)
cols_to_fix = ['startYear', 'runtimeMinutes', 'numVotes']

for col in cols_to_fix:
    v3_update_omdb[col] = pd.to_numeric(v3_update_omdb[col], errors='coerce').astype('Int64')

# List of columns that should be Integers (Int64)
int_cols_to_check = ['startYear', 'runtimeMinutes', 'numVotes']

print("--- Data Type Verification for Integer Columns ---")

for col in int_cols_to_check:
    # Accessing the first value in the column (index 0)
    first_val = v3_update_omdb[col].iloc[0]
    
    # Printing the result in a clear, single line per column
    print(f"Deep Check on '{col}' column - Type of the first value: {type(first_val)}")

--- Data Type Verification for Integer Columns ---
Deep Check on 'startYear' column - Type of the first value: <class 'numpy.int64'>
Deep Check on 'runtimeMinutes' column - Type of the first value: <class 'numpy.int64'>
Deep Check on 'numVotes' column - Type of the first value: <class 'numpy.int64'>


**Float Verification**

Following the project requirements, the 'averageRating' column is verified as a Float type.

In [139]:
# Check the column that should be Float
float_col_to_check = ['averageRating']

print("--- Data Type Verification for Float Column ---")

for col in float_col_to_check:
    # Accessing the first value in the column (index 0)
    first_val = v3_update_omdb[col].iloc[0]
    
    # Printing the result in a clear, single line
    print(f"Deep Check on '{col}' column - Type of the first value: {type(first_val)}")

--- Data Type Verification for Float Column ---
Deep Check on 'averageRating' column - Type of the first value: <class 'numpy.float64'>


**Currency Collection & Conversion**

To meet the project requirements, all currency symbols and formats present in the dataset were identified. Immediately following this collection, the values are converted into a unified metric of Millions of USD and assigned to the designated columns. This ensures all financial data is standardized and ready for analysis.

In [140]:
# Function to extract text labels (currencies and units) from a column
def get_all_labels(df, column_name):
    # Get all unique non-null values
    unique_vals = df[column_name].dropna().unique()
    
    all_labels = set()
    for val in unique_vals:
        # Remove numbers, dots, commas, hyphens, and spaces to isolate text
        label = re.sub(r'[\d\.\,\-\s]', '', str(val))
        if label:
            all_labels.add(label)
    
    # Return all unique labels as a single comma-separated string
    return ", ".join(all_labels)

# Execute and print the results for the budget column
result = get_all_labels(v3_update_omdb, 'budget')
print("--- Copy the following list: ---")
print(result)

--- Copy the following list: ---
<$million, ~$million, $million(₽million), DKKmillion(estimated), ¥million, SovietRuble, RM, $million(USD), lakhsINR, R, $million(NorthAmerica)milliontickets(worldwide), €million(gross)(~$million)$–million(net), $million$or$, $est, £or£, millionpesetas, Crore, ₹–crore, under$millionor$, $million(€million), ₹lakh(US$), DM, Rbls, FIMmillion, ₹crores, ₤million(approximately$), $million(est), $million(approx), ₹million(US$), MillionKČs, £million, est₹–₹crore, $, A$, ₽million$million, €million(≈$million), $million, £(est), ₹crore, S$, belowUS$, ₩million(~US$), millionℛℳ, ₹–crore(sharedwithPart), $–million, FIM, ₹crore(US$), ₴, $M, $–, ₹, est₹crore, CAD$million, $million(₽billion), €million, ₹million, ₹Lakh, £million($million), EGPmillion, £millionor$million, MYRmillion, £, croreLKR, est₹lakhs, ₴million, >$, ₹–Crore, A$million(est), ₹lakh(est₹croreasof), $(estimated), $Million, ₱million, <$, over$million, €million($million), HK$, millionkróna(estimated), ₹lakh

In [141]:

def final_currency_to_usd_millions(value):
    # Check for null values or empty strings
    if pd.isna(value) or value == 'N/A' or str(value).strip() == '':
        return np.nan
    
    # Initial cleaning: lowercase and normalize dashes
    val_str = str(value).lower().replace(',', '').replace('–', '-').replace('—', '-').strip()
    raw_original = val_str 
    
    # Priority: If there is a dollar amount inside parentheses, extract it
    if '(' in val_str and '$' in val_str:
        bracket_content = re.search(r'\((.*?)\)', val_str)
        if bracket_content and '$' in bracket_content.group(1):
            val_str = bracket_content.group(1)

    # Conversion rates to USD
    rates = {'£': 1.25, '₤': 1.25, '€': 1.10, '₹': 0.012, 'inr': 0.012, '₽': 0.011, 'rs.': 0.012}
    target_num = None

    def clean_extract(text):
        # Fix for ranges like 70-80000:
        # Search for two numbers separated by a dash
        range_match = re.search(r'(\d[\d\s\.]*)\s*-\s*(\d[\d\s\.]*)', text)
        if range_match:
            num1_str = range_match.group(1).replace(' ', '')
            num2_str = range_match.group(2).replace(' ', '')
            
            n1 = float(num1_str)
            n2 = float(num2_str)
            
            # Logic for shorthand ranges (e.g., 70-80k becomes 70k-80k)
            if n1 < 1000 and n2 >= 1000:
                if n2 >= 10000: n1 *= 1000
                elif n2 >= 1000: n1 *= 100
            
            return (n1 + n2) / 2
        
        # Search for a single number if no range is found
        num_match = re.search(r'(\d[\d\s\.]*)', text)
        if num_match:
            num_txt = num_match.group(1).replace(' ', '').rstrip('.')
            try: return float(num_txt)
            except: return None
        return None

    # Search for Dollar patterns first
    usd_pattern = re.compile(r'[^₽£€₹]*(\$|usd|us\$)[^₽£€₹]*')
    usd_search = usd_pattern.search(val_str)
    
    if usd_search:
        target_num = clean_extract(usd_search.group(0))
        val_str = usd_search.group(0)
    else:
        # If no dollar, check for other currencies in the rates dictionary
        for symbol, rate in rates.items():
            if symbol in val_str:
                target_num = clean_extract(val_str)
                if target_num:
                    target_num = target_num * rate
                    break
        # Fallback to general number extraction if no currency symbol found
        if target_num is None:
            target_num = clean_extract(val_str)

    if target_num is None: return np.nan

    # Identify if the context implies Millions, Billions, etc.
    is_million_context = any(unit in val_str for unit in ['million', 'billion', 'crore', 'lakh', ' m ']) or val_str.strip().endswith('m')

    # Normalization: If it's a large number without a "million" context, divide by 1M
    if target_num >= 100 and not is_million_context:
        final_val = target_num / 1_000_000
    else:
        final_val = target_num

    # Apply multipliers based on identified keywords
    if is_million_context:
        if 'billion' in val_str: final_val *= 1000
        elif 'crore' in val_str: final_val *= 10
        elif 'lakh' in val_str: final_val *= 0.1

    # Safety Control: If result is suspicious (>100) and raw text contains a $, re-extract directly
    if final_val > 100 and '$' in raw_original:
        usd_direct = re.search(r'(?:\$|usd)\s*(\d[\d\s\.]*)|(\d[\d\s\.]*)\s*(?:\$|usd)', raw_original)
        if usd_direct:
            num_part = usd_direct.group(1) if usd_direct.group(1) else usd_direct.group(2)
            fixed_num = clean_extract(num_part)
            if fixed_num:
                if fixed_num < 100: final_val = fixed_num
                else: final_val = fixed_num / 1_000_000

    return round(final_val, 3)

In [142]:
# --- STEP 0: Create the RAW columns first ---
# We take the current columns (which still have symbols like $, £, etc.) 
# and save them as 'budget_raw' and 'BoxOffice_raw'
v3_update_omdb['budget_raw'] = v3_update_omdb['budget']
v3_update_omdb['BoxOffice_raw'] = v3_update_omdb['BoxOffice']

# --- STEP 1: Now you can run the conversion safely ---
# This takes the text from 'raw', cleans it, and puts the number back in 'budget'
v3_update_omdb['budget'] = v3_update_omdb['budget_raw'].apply(final_currency_to_usd_millions)
v3_update_omdb['BoxOffice'] = v3_update_omdb['BoxOffice_raw'].apply(final_currency_to_usd_millions)

# --- STEP 2: Verification ---
print("--- Conversion Completed Successfully ---")
cols_to_check = ['primaryTitle', 'budget_raw', 'budget', 'BoxOffice_raw', 'BoxOffice']
display(v3_update_omdb[v3_update_omdb['budget'].notna()][cols_to_check].head(10))

--- Conversion Completed Successfully ---


,primaryTitle,budget_raw,budget,BoxOffice_raw,BoxOffice
19,Voices of the City,"$200,000",0.200,NaN,NaN
68,Untamed,"$229,000",0.229,"$974,000",0.974
85,Viennese Nights,"$604,000",0.604,"$950,000",0.950
107,Union Depot,"$284,000",0.284,"$637,000[A]",0.637
127,Vagabond Violinist,"£20,000 (est.)",0.025,NaN,NaN
136,Viva Villa!,"$1,022,000",1.022,"$1,969,000 (worldwide rentals)",1.969
138,Villa for Sale,"$111,000",0.111,NaN,NaN
155,Under Two Flags,"$1,250,000",1.250,NaN,NaN
157,Undersea Kingdom,"$81,924 ( negative cost : $99,222)",0.099,NaN,NaN
169,Varsity Show,over $1 million,1.000,NaN,NaN


In [145]:
# Create a table containing only the audit columns
conversion_audit = v3_update_omdb[['primaryTitle', 'budget_raw', 'budget', 'BoxOffice_raw', 'BoxOffice']]

# Export to Excel - all rows
conversion_audit.to_excel('currency_conversion_audit24.xlsx', index=False)

print("Audit file created successfully!")

Audit file created successfully!


In [146]:
print("--- Data Type Verification for Float Column ---")
print(f"Budget type: {type(v3_update_omdb['budget'].dropna().iloc[0])}")
print(f"BoxOffice type: {type(v3_update_omdb['BoxOffice'].dropna().iloc[0])}")

--- Data Type Verification for Float Column ---
Budget type: <class 'numpy.float64'>
BoxOffice type: <class 'numpy.float64'>


**Final Dataset Consolidation**

To finalize the data preparation phase, all processed columns are consolidated into a single table. This ensures that the dataset contains only the 13 required fields, each formatted in its specified data type, resulting in a clean and structured final table.

In [147]:
# List of the final 13 columns we want to keep
final_columns = [
    'tconst', 'primaryTitle', 'Language', 'Country', 'plot',       
    'genres', 'lead_actors_ids',                                  
    'startYear', 'runtimeMinutes', 'numVotes',                    
    'averageRating', 'budget', 'BoxOffice'                        
]

# Keep only the selected columns
v3_update_omdb = v3_update_omdb[final_columns]

# Verification: Print the final column list and the first 5 rows
print("--- Final Dataset Ready (Raw columns removed) ---")
print(f"Total columns: {len(v3_update_omdb.columns)}")
v3_update_omdb.head()

--- Final Dataset Ready (Raw columns removed) ---
Total columns: 13


,tconst,primaryTitle,Language,Country,plot,genres,lead_actors_ids,startYear,runtimeMinutes,numVotes,averageRating,budget,BoxOffice
0,tt0006709,Under Suspicion,Silent,United States,"As described in afilm magazine,[3]Gerry Simpso...","[Comedy, Crime]","[nm0024706, nm0613115, nm0184766, nm0944017...",1916,60,34,7.2,NaN,NaN
1,tt0006903,Victory of Love,NaN,NaN,NaN,[Drama],"[nm0436013, nm0459320, nm0054011, nm0188850...",1916,62,<NA>,NaN,NaN,NaN
2,tt0007498,Under Two Flags,Silent,United States,"As described in a film magazine,[3]British nob...","[Adventure, Drama]","[nm0000847, nm0382229, nm0392059, nm0923657...",1916,60,30,6.5,NaN,NaN
3,tt0007522,Vingarne,Silent,Sweden,"The story is that of a conniving countess, Luc...",[Drama],"[nm0251622, nm0064949, nm0361319, nm0492280...",1916,69,230,5.7,NaN,NaN
4,tt0009742,Under Four Flags,NaN,NaN,NaN,"[Documentary, War]",NaN,1918,70,<NA>,NaN,NaN,NaN


In [148]:
# --- COMPACT DATA COMPLETENESS AUDIT ---

# 0. Pre-processing: Treat empty lists or empty strings in specific columns as Null
# This ensures genres and actors are counted correctly
df_audit = v3_update_omdb.copy()

# 1. Column-wise detailed analysis (using the fixed dataframe)
null_counts = df_audit.isnull().sum()
full_counts = df_audit.notna().sum()
null_percentages = df_audit.isnull().mean() * 100

# Create detailed report table
null_report = pd.DataFrame({
    'Column Name': null_counts.index,
    'Empty Cells (Null)': null_counts.values,
    'Full Cells (Valid)': full_counts.values,
    'Null Percentage (%)': null_percentages.values
})
null_report = null_report.sort_values(by='Null Percentage (%)', ascending=False)

# 2. Calculate summary metrics
total_cells = df_audit.size
total_empty = df_audit.isnull().sum().sum()
total_full = df_audit.notna().sum().sum()
overall_null_pct = (total_empty / total_cells) * 100
overall_full_pct = 100 - overall_null_pct

# Extraction metrics (5 columns)
extra_cols = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']
ext_df = df_audit[extra_cols]
ext_total = ext_df.size
ext_nulls = ext_df.isnull().sum().sum()
ext_full = ext_df.notna().sum().sum()
ext_null_pct = (ext_nulls / ext_total) * 100
ext_full_pct = 100 - ext_null_pct

# --- PRINTING REPORTS ---

print("--- DETAILED PER-COLUMN COMPLETENESS (Cleaned) ---")
print(null_report.to_string(index=False, formatters={'Null Percentage (%)': '{:.2f}%'.format}))

summary_data = {
    "Metric": ["Total Cells", "Empty (Null)", "Full (Valid)", "Null %", "Full %"],
    "Extraction (5 Cols)": [f"{ext_total:,}", f"{ext_nulls:,}", f"{ext_full:,}", f"{ext_null_pct:.2f}%", f"{ext_full_pct:.2f}%"],
    "Overall (13 Cols)": [f"{total_cells:,}", f"{total_empty:,}", f"{total_full:,}", f"{overall_null_pct:.2f}%", f"{overall_full_pct:.2f}%"]
}
summary_df = pd.DataFrame(summary_data)

print("\n" + "="*60)
print("--- FINAL DATASET AUDIT SUMMARY (Cleaned) ---")
print(summary_df.to_string(index=False))
print("="*60)

--- DETAILED PER-COLUMN COMPLETENESS (Cleaned) ---
    Column Name  Empty Cells (Null)  Full Cells (Valid) Null Percentage (%)
         budget               10542                 579              94.79%
      BoxOffice               10372                 749              93.26%
           plot                8302                2819              74.65%
        Country                8286                2835              74.51%
       Language                8198                2923              73.72%
       numVotes                3314                7807              29.80%
  averageRating                3314                7807              29.80%
lead_actors_ids                1856                9265              16.69%
         genres                 669               10452               6.02%
         tconst                   0               11121               0.00%
   primaryTitle                   0               11121               0.00%
      startYear                   0  

**Final Data Quality & Integration Audit**

This final report provides a transparent overview of the dataset's status following the collection and enrichment phases. The audit distinguishes between the targeted extraction efforts and the overall final structure:

* **Extraction Success (Wikipedia & OMDB):** Focused on the 5 enriched columns (Language, Country, Budget, Box Office, and Plot), the process successfully recovered and populated nearly 10,000 data points. This significantly reduces the gaps in these challenging financial and descriptive fields.
* **Overall Dataset Integrity:** The final consolidated table maintains a solid fill rate, with  62.06% of all cells across the 13 columns containing valid data. This ensures a consistent and reliable foundation, with all fields correctly typed and formatted.

This completion marks the end of the data preparation phase. The dataset is now fully structured and verified, ready for any further requirements.

In [149]:
# 1. Create a helper column to count missing values per row
v3_update_omdb['null_count'] = v3_update_omdb.isnull().sum(axis=1)

# 2. Select the top 5,000 most complete rows (fewest nulls first)
best_5000_df = v3_update_omdb.sort_values(by='null_count', ascending=True).head(5000).copy()

# 3. Calculate statistics for the selected 5,000 rows
total_cells_5000 = best_5000_df.drop(columns=['null_count']).size
total_nulls_5000 = best_5000_df.drop(columns=['null_count']).isnull().sum().sum()
null_pct_5000 = (total_nulls_5000 / total_cells_5000) * 100
full_pct_5000 = 100 - null_pct_5000

# 4. Generate a quality breakdown report by "null tiers"
quality_breakdown = best_5000_df['null_count'].value_counts().sort_index()

print("="*60)
print("--- FINAL 5,000 ROWS QUALITY REPORT ---")
print(f"Total Null Percentage in this set:  {null_pct_5000:.2f}%")
print(f"Total Full Percentage in this set:  {full_pct_5000:.2f}%")
print("-" * 60)

print("--- ROWS BREAKDOWN BY MISSING VALUES ---")
for nulls, count in quality_breakdown.items():
    completeness = 13 - nulls
    print(f"Rows with {nulls} missing values ({completeness}/13 full): {count:,}")

print("-" * 60)

# 5. Export the final curated dataset to CSV (removing the helper column)
final_export = best_5000_df.drop(columns=['null_count'])
final_export.to_csv('movies_dataset_final_5000.csv', index=False)

print(f"SUCCESS: Saved 5,000 rows to 'movies_dataset_final_5000.csv'")
print("="*60)

--- FINAL 5,000 ROWS QUALITY REPORT ---
Total Null Percentage in this set:  24.64%
Total Full Percentage in this set:  75.36%
------------------------------------------------------------
--- ROWS BREAKDOWN BY MISSING VALUES ---
Rows with 0 missing values (13/13 full): 364
Rows with 1 missing values (12/13 full): 402
Rows with 2 missing values (11/13 full): 1,570
Rows with 3 missing values (10/13 full): 284
Rows with 4 missing values (9/13 full): 276
Rows with 5 missing values (8/13 full): 2,104
------------------------------------------------------------
SUCCESS: Saved 5,000 rows to 'movies_dataset_final_5000.csv'


In [150]:
# --- COMPACT DATA COMPLETENESS AUDIT FOR THE FINAL 5000 DATASET (SAFE VERSION) ---

# 0. Pre-processing: Create a dedicated copy for the audit to avoid overwriting

f5k_audit_df = pd.read_csv("movies_dataset_final_5000.csv") 

# 1. Column-wise detailed analysis for the 5000 subset
f5k_null_counts = f5k_audit_df.isnull().sum()
f5k_full_counts = f5k_audit_df.notna().sum()
f5k_null_percentages = f5k_audit_df.isnull().mean() * 100

f5k_null_report = pd.DataFrame({
    'Column Name': f5k_null_counts.index,
    'Empty Cells (Null)': f5k_null_counts.values,
    'Full Cells (Valid)': f5k_full_counts.values,
    'Null Percentage (%)': f5k_null_percentages.values
})
f5k_null_report = f5k_null_report.sort_values(by='Null Percentage (%)', ascending=False)

# 2. Calculate summary metrics for the final 5000
f5k_total_cells = f5k_audit_df.size
f5k_total_empty = f5k_audit_df.isnull().sum().sum()
f5k_total_full = f5k_audit_df.notna().sum().sum()
f5k_overall_null_pct = (f5k_total_empty / f5k_total_cells) * 100
f5k_overall_full_pct = 100 - f5k_overall_null_pct

# Metrics for the 5 extra columns (Extraction)
f5k_extra_cols = ['Language', 'Country', 'budget', 'BoxOffice', 'plot']
f5k_ext_df = f5k_audit_df[f5k_extra_cols]
f5k_ext_total = f5k_ext_df.size
f5k_ext_nulls = f5k_ext_df.isnull().sum().sum()
f5k_ext_full = f5k_ext_df.notna().sum().sum()
f5k_ext_null_pct = (f5k_ext_nulls / f5k_ext_total) * 100
f5k_ext_full_pct = 100 - f5k_ext_null_pct

# --- PRINTING THE FINAL REPORT ---

print("--- DETAILED PER-COLUMN COMPLETENESS (FINAL 5000 SUBSET) ---")
print(f5k_null_report.to_string(index=False, formatters={'Null Percentage (%)': '{:.2f}%'.format}))

f5k_summary_data = {
    "Metric": ["Total Cells", "Empty (Null)", "Full (Valid)", "Null %", "Full %"],
    "Extraction (5 Cols)": [f"{f5k_ext_total:,}", f"{f5k_ext_nulls:,}", f"{f5k_ext_full:,}", f"{f5k_ext_null_pct:.2f}%", f"{f5k_ext_full_pct:.2f}%"],
    "Overall (13 Cols)": [f"{f5k_total_cells:,}", f"{f5k_total_empty:,}", f"{f5k_total_full:,}", f"{f5k_overall_null_pct:.2f}%", f"{f5k_overall_full_pct:.2f}%"]
}
f5k_summary_df = pd.DataFrame(f5k_summary_data)

print("\n" + "="*60)
print("--- FINAL 5000 DATASET AUDIT SUMMARY ---")
print(f5k_summary_df.to_string(index=False))
print("="*60)

--- DETAILED PER-COLUMN COMPLETENESS (FINAL 5000 SUBSET) ---
    Column Name  Empty Cells (Null)  Full Cells (Valid) Null Percentage (%)
         budget                4423                 577              88.46%
      BoxOffice                4253                 747              85.06%
        Country                2247                2753              44.94%
           plot                2217                2783              44.34%
       Language                2170                2830              43.40%
       numVotes                 244                4756               4.88%
  averageRating                 244                4756               4.88%
lead_actors_ids                 154                4846               3.08%
         genres                  66                4934               1.32%
         tconst                   0                5000               0.00%
   primaryTitle                   0                5000               0.00%
      startYear            